# Post 007 — Anomaly Detection & Dimensionality Reduction
## Dataset B: Silicon Thermal Hotspot Detection (Post-Silicon Validation)

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

During post-silicon validation, thermal sensors monitor chip temperature across hundreds of locations. Most readings are within normal operating range. But occasionally, a localized hotspot appears — a sign of unexpected power density, a failing power domain, or a routing issue that slipped through simulation.

This notebook applies **Isolation Forest** and **LOF** to detect thermal anomalies in silicon, combined with **PCA** to understand which thermal zones are most correlated. The result: automated hotspot detection that flags issues before they cause chip damage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/silicon_thermal_hotspots.csv')
print(f'Shape: {df.shape}')
print(f'\nAnomaly distribution:')
print(df['is_hotspot'].value_counts())
print(f'Hotspot rate: {df["is_hotspot"].mean():.1%}')
df.head()

## 1. Thermal Heatmap Visualization

Before applying any algorithm, let's visualize the thermal data as a chip heatmap. This is how a validation engineer would normally inspect thermal data — but manually, one snapshot at a time. Our goal is to automate this across thousands of measurements.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['is_hotspot', 'sensor_id', 'timestamp']]
X = df[feature_cols].values
y_true = df['is_hotspot'].values

# Show mean temperature profile for normal vs hotspot readings
normal_mean = df.loc[df['is_hotspot']==0, feature_cols].mean()
hotspot_mean = df.loc[df['is_hotspot']==1, feature_cols].mean()
diff = hotspot_mean - normal_mean

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(range(len(feature_cols)), normal_mean, color='steelblue', alpha=0.7)
axes[0].set_title('Normal Reading: Mean Temperatures')
axes[0].set_xlabel('Thermal Zone'); axes[0].set_ylabel('Temperature (°C)')
axes[0].set_xticks(range(len(feature_cols)))
axes[0].set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)

axes[1].bar(range(len(feature_cols)), hotspot_mean, color='red', alpha=0.7)
axes[1].set_title('Hotspot Reading: Mean Temperatures')
axes[1].set_xlabel('Thermal Zone')
axes[1].set_xticks(range(len(feature_cols)))
axes[1].set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)

colors_diff = ['red' if d > 0 else 'steelblue' for d in diff]
axes[2].bar(range(len(feature_cols)), diff, color=colors_diff, alpha=0.7)
axes[2].axhline(y=0, color='black', linewidth=0.5)
axes[2].set_title('Temperature Delta: Hotspot - Normal')
axes[2].set_xlabel('Thermal Zone')
axes[2].set_xticks(range(len(feature_cols)))
axes[2].set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

## 2. PCA: Understanding Thermal Correlations

Thermal zones on a chip are highly correlated — the CPU core heats up the surrounding cache, which heats the memory controller, and so on. PCA reveals these correlation structures and identifies which zones are the primary drivers of variance.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative variance
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
ax1.plot(range(1, len(cum_var)+1), cum_var, 'bo-', linewidth=2)
ax1.axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
ax1.set_xlabel('Number of Components')
ax1.set_ylabel('Cumulative Explained Variance')
ax1.set_title('PCA: How many components capture 95% of thermal variance?')
ax1.legend()

# PCA space colored by hotspot
for label, color, name, size in [(0, 'steelblue', 'Normal', 8), (1, 'red', 'Hotspot', 40)]:
    mask = y_true == label
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, 
               alpha=0.3 if label==0 else 0.8, s=size)
ax2.set_title(f'PCA Space: Normal vs Hotspot ({pca2.explained_variance_ratio_.sum():.1%} variance)')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.legend()

plt.tight_layout()
plt.show()

n_for_95 = (cum_var < 0.95).sum() + 1
print(f'Components needed for 95% variance: {n_for_95} out of {len(feature_cols)}')

## 3. Isolation Forest for Hotspot Detection

Isolation Forest is particularly well-suited for silicon thermal data because:
1. Hotspots are rare (low contamination rate)
2. The data is high-dimensional (many thermal zones)
3. We have no labeled training examples of what a hotspot looks like in advance

In [ ]:
contamination_rate = y_true.mean()
print(f'True contamination rate: {contamination_rate:.3f}')

iso = IsolationForest(contamination=contamination_rate, random_state=42, n_estimators=200)
iso_pred = iso.fit_predict(X_scaled)
iso_binary = (iso_pred == -1).astype(int)
iso_scores = -iso.score_samples(X_scaled)

print('\nIsolation Forest Results:')
print(classification_report(y_true, iso_binary, target_names=['Normal', 'Hotspot']))
print(f'ROC-AUC: {roc_auc_score(y_true, iso_scores):.3f}')

In [ ]:
# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination_rate)
lof_pred = lof.fit_predict(X_scaled)
lof_binary = (lof_pred == -1).astype(int)
lof_scores = -lof.negative_outlier_factor_

print('Local Outlier Factor Results:')
print(classification_report(y_true, lof_binary, target_names=['Normal', 'Hotspot']))
print(f'ROC-AUC: {roc_auc_score(y_true, lof_scores):.3f}')

## 4. Ensemble: Combining Both Detectors

In practice, combining multiple anomaly detectors often outperforms any single method. A simple ensemble: flag a point as anomalous only if **both** detectors agree. This reduces false positives at the cost of some recall.

In [ ]:
# Ensemble: both must agree
ensemble_pred = ((iso_binary == 1) & (lof_binary == 1)).astype(int)
ensemble_score = (iso_scores / iso_scores.max() + lof_scores / lof_scores.max()) / 2

print('Ensemble (IF AND LOF) Results:')
print(classification_report(y_true, ensemble_pred, target_names=['Normal', 'Hotspot']))
print(f'ROC-AUC: {roc_auc_score(y_true, ensemble_score):.3f}')

# Comparison table
from sklearn.metrics import precision_score, recall_score, f1_score
results = pd.DataFrame({
    'Method': ['Isolation Forest', 'LOF', 'Ensemble'],
    'Precision': [
        precision_score(y_true, iso_binary),
        precision_score(y_true, lof_binary),
        precision_score(y_true, ensemble_pred)
    ],
    'Recall': [
        recall_score(y_true, iso_binary),
        recall_score(y_true, lof_binary),
        recall_score(y_true, ensemble_pred)
    ],
    'F1': [
        f1_score(y_true, iso_binary),
        f1_score(y_true, lof_binary),
        f1_score(y_true, ensemble_pred)
    ],
    'ROC-AUC': [
        roc_auc_score(y_true, iso_scores),
        roc_auc_score(y_true, lof_scores),
        roc_auc_score(y_true, ensemble_score)
    ]
})
print('\nMethod Comparison:')
print(results.round(3).to_string(index=False))

## 5. Summary

This notebook showed how unsupervised anomaly detection can automatically flag thermal hotspots in silicon validation data — without any labeled training examples.

**Engineering impact**: Instead of a validation engineer manually reviewing thousands of thermal snapshots, the algorithm flags the top 3-5% for human review. This is a 20x reduction in manual inspection time.

**When to use which method:**
- **Isolation Forest**: Large datasets (>10K rows), streaming data, unknown anomaly types
- **LOF**: Smaller datasets, when anomalies are embedded within dense normal regions
- **Ensemble**: Production systems where false positives are costly (e.g., triggering unnecessary chip re-spins)